# 04 — Cross-File Generalization

Two experiments testing how models generalize to unseen attack types.

**Experiment A** — Train: Web + DDoS  →  Test: PortScan

**Experiment B** — Train: All except Infiltration  →  Test: Infiltration

In [1]:
%run data_loader.ipynb

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, confusion_matrix
)

Config loaded
preprocess_binary() defined
load_all_datasets() defined
Project root:          /Users/yp_home/Documents/IDS_project
CICIDS2018 folder exists: True
load_cicids2018() defined


## 1. Load All Datasets

In [2]:
datasets = load_all_datasets()
print(f"Loaded: {list(datasets.keys())}")

Loading 8 dataset(s): ['monday', 'bruteforce', 'dos', 'web_attacks', 'infiltration', 'botnet', 'portscan', 'ddos']

  [monday]  Monday-WorkingHours.pcap_ISCX.csv
           raw (529918, 79)  ->  clean (502650, 81)  (attack rate 0.0%)
  [bruteforce]  Tuesday-WorkingHours.pcap_ISCX.csv
           raw (445909, 79)  ->  clean (421626, 81)  (attack rate 2.2%)
  [dos]  Wednesday-workingHours.pcap_ISCX.csv
           raw (692703, 79)  ->  clean (610492, 81)  (attack rate 31.7%)
  [web_attacks]  Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv
           raw (170366, 79)  ->  clean (164179, 81)  (attack rate 1.3%)
  [infiltration]  Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv
           raw (288602, 79)  ->  clean (252790, 81)  (attack rate 0.0%)
  [botnet]  Friday-WorkingHours-Morning.pcap_ISCX.csv
           raw (191033, 79)  ->  clean (184044, 81)  (attack rate 1.1%)
  [portscan]  Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv
           raw (286467, 79)  ->  clea

## 2. Helper

In [3]:
def run_experiment(train_df, test_df, experiment_name):
    print(f"\n{'='*60}")
    print(f"  {experiment_name}")
    print(f"  Train: {len(train_df):,}   Test: {len(test_df):,}")
    print(f"  Test attack rate: {test_df['Label_Binary'].mean()*100:.1f}%")
    print(f"{'='*60}")

    drop_cols = ["Label", "Label_Binary", "Source_File"]
    X_train = train_df.drop(columns=drop_cols).replace([np.inf, -np.inf], np.nan)
    X_test  = test_df.drop(columns=drop_cols).replace([np.inf, -np.inf], np.nan)
    y_train = train_df["Label_Binary"]
    y_test  = test_df["Label_Binary"]

    medians = X_train.median(numeric_only=True)
    X_train = X_train.fillna(medians)
    X_test  = X_test.fillna(medians)

    scaler = StandardScaler()
    X_train_sc = scaler.fit_transform(X_train)
    X_test_sc  = scaler.transform(X_test)

    results = []
    for name, model, Xtr, Xte in [
        ("Logistic Regression", LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42), X_train_sc, X_test_sc),
        ("Random Forest",       RandomForestClassifier(n_estimators=100, class_weight="balanced", n_jobs=-1, random_state=42), X_train, X_test),
    ]:
        model.fit(Xtr, y_train)
        y_pred = model.predict(Xte)
        y_prob = model.predict_proba(Xte)[:, 1]
        print(f"\n{name} — Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
        results.append({
            "Model":     name,
            "Accuracy":  accuracy_score(y_test, y_pred),
            "Precision": precision_score(y_test, y_pred, zero_division=0),
            "Recall":    recall_score(y_test, y_pred, zero_division=0),
            "F1":        f1_score(y_test, y_pred, zero_division=0),
            "ROC_AUC":   roc_auc_score(y_test, y_prob),
        })

    return pd.DataFrame(results)

## 3. Experiment A — Web + DDoS  →  PortScan

In [4]:
train_A = pd.concat([datasets["web_attacks"], datasets["ddos"]], ignore_index=True)
results_A = run_experiment(train_A, datasets["portscan"], "Exp A: Web+DDoS → PortScan")
results_A.round(4)


  Exp A: Web+DDoS → PortScan
  Train: 387,261   Test: 213,777
  Test attack rate: 42.4%

Logistic Regression — Confusion Matrix:
 [[119656   3427]
 [ 90553    141]]

Random Forest — Confusion Matrix:
 [[123074      9]
 [ 90626     68]]


,Model,Accuracy,Precision,Recall,F1,ROC_AUC
0,Logistic Regression,0.5604,0.0395,0.0016,0.0030,0.7211
1,Random Forest,0.5760,0.8831,0.0007,0.0015,0.6707


## 4. Experiment B — All Except Infiltration  →  Infiltration

In [5]:
train_B = pd.concat([v for k, v in datasets.items() if k != "infiltration"], ignore_index=True)
results_B = run_experiment(train_B, datasets["infiltration"], "Exp B: All → Infiltration")
results_B.round(4)


  Exp B: All → Infiltration
  Train: 2,319,850   Test: 252,790
  Test attack rate: 0.0%

Logistic Regression — Confusion Matrix:
 [[202503  50251]
 [    26     10]]

Random Forest — Confusion Matrix:
 [[251666   1088]
 [    36      0]]


,Model,Accuracy,Precision,Recall,F1,ROC_AUC
0,Logistic Regression,0.8011,0.0002,0.2778,0.0004,0.3344
1,Random Forest,0.9956,0.0000,0.0000,0.0000,0.6396


## 5. Side-by-Side Comparison

In [6]:
results_A["Experiment"] = "A: Web+DDoS → PortScan"
results_B["Experiment"] = "B: All → Infiltration"
pd.concat([results_A, results_B])[["Experiment","Model","Accuracy","Precision","Recall","F1","ROC_AUC"]].round(4)

,Experiment,Model,Accuracy,Precision,Recall,F1,ROC_AUC
0,A: Web+DDoS → PortScan,Logistic Regression,0.5604,0.0395,0.0016,0.0030,0.7211
1,A: Web+DDoS → PortScan,Random Forest,0.5760,0.8831,0.0007,0.0015,0.6707
0,B: All → Infiltration,Logistic Regression,0.8011,0.0002,0.2778,0.0004,0.3344
1,B: All → Infiltration,Random Forest,0.9956,0.0000,0.0000,0.0000,0.6396


---
## 6. Cross-Year Generalization: CICIDS2017 → CSE-CIC-IDS2018

Train on all CICIDS2017 files, test on CICIDS2018.
Feature alignment is required because column names differ slightly between years.

In [7]:
def align_to_train(df_test, train_feature_cols):
    """
    Align a test DataFrame's columns to match training feature columns exactly.
    - Strips whitespace from column names
    - Adds missing columns as 0
    - Drops extra columns
    - Returns DataFrame with columns in exact training order
    """
    df = df_test.copy()
    df.columns = df.columns.str.strip()

    missing = set(train_feature_cols) - set(df.columns)
    extra   = set(df.columns) - set(train_feature_cols)

    if missing:
        print(f'  Adding {len(missing)} missing columns as 0')
        for col in missing:
            df[col] = 0
    if extra:
        print(f'  Dropping {len(extra)} extra columns')

    df = df[train_feature_cols]
    df = df.apply(pd.to_numeric, errors='coerce').fillna(0)
    return df


def run_cross_year_experiment(train_df, test_df, experiment_name):
    """
    Same as run_experiment but handles feature alignment between datasets.
    """
    print(f"\n{'='*60}")
    print(f"  {experiment_name}")
    print(f"  Train: {len(train_df):,}   Test: {len(test_df):,}")
    attack_rate = test_df['Label_Binary'].mean()
    print(f"  Test attack rate: {attack_rate*100:.1f}%")
    print(f"{'='*60}")

    if attack_rate < 0.001:
        print('  WARNING: Too few attacks for reliable evaluation. Skipping.')
        return None

    drop_cols = ['Label', 'Label_Binary', 'Source_File', 'SourceFile']

    # Build training features
    X_train = train_df.drop(columns=[c for c in drop_cols if c in train_df.columns])
    X_train = X_train.replace([np.inf, -np.inf], np.nan)
    medians  = X_train.median(numeric_only=True)
    X_train  = X_train.fillna(medians)
    y_train  = train_df['Label_Binary']
    train_feature_cols = X_train.columns.tolist()

    # Align test features to training columns
    X_test_raw = test_df.drop(columns=[c for c in drop_cols if c in test_df.columns])
    X_test = align_to_train(X_test_raw, train_feature_cols)
    X_test = X_test.replace([np.inf, -np.inf], np.nan).fillna(medians)
    y_test = test_df['Label_Binary']

    print(f'  Train features: {X_train.shape[1]}   Test features after alignment: {X_test.shape[1]}')

    scaler     = StandardScaler()
    X_train_sc = scaler.fit_transform(X_train)
    X_test_sc  = scaler.transform(X_test)

    results = []
    for name, model, Xtr, Xte in [
        ('Logistic Regression',
         LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42),
         X_train_sc, X_test_sc),
        ('Random Forest',
         RandomForestClassifier(n_estimators=100, class_weight='balanced', n_jobs=-1, random_state=42),
         X_train, X_test),
    ]:
        model.fit(Xtr, y_train)
        y_pred = model.predict(Xte)
        y_prob = model.predict_proba(Xte)[:, 1]
        print(f'\n  {name} Confusion Matrix:\n', confusion_matrix(y_test, y_pred))
        results.append({
            'Model':     name,
            'Accuracy':  round(accuracy_score(y_test, y_pred), 4),
            'Precision': round(precision_score(y_test, y_pred, zero_division=0), 4),
            'Recall':    round(recall_score(y_test, y_pred, zero_division=0), 4),
            'F1':        round(f1_score(y_test, y_pred, zero_division=0), 4),
            'ROC_AUC':   round(roc_auc_score(y_test, y_prob), 4),
        })

    return pd.DataFrame(results)

print('run_cross_year_experiment() defined')

run_cross_year_experiment() defined


In [8]:
df_2018 = load_cicids2018(max_rows_per_file=30000)

if df_2018 is not None:
    train_2017 = pd.concat(datasets.values(), ignore_index=True)
    results_cross_year = run_cross_year_experiment(
        train_2017, df_2018,
        'Cross-Year: All CICIDS2017 → CSE-CIC-IDS2018'
    )
    if results_cross_year is not None:
        print('\nResults:')
        print(results_cross_year.to_string(index=False))
else:
    print('CICIDS2018 not available. Add CSVs to data/cse_cic_ids2018/')

Found 10 CSE-CIC-IDS2018 files
  Loading: 02-14-2018.csv
  Loading: 02-15-2018.csv
  Loading: 02-16-2018.csv
  Loading: 02-20-2018.csv
  Loading: 02-21-2018.csv
  Loading: 02-22-2018.csv
  Loading: 02-23-2018.csv
  Loading: 02-28-2018.csv
  Loading: 03-01-2018.csv
  Loading: 03-02-2018.csv

2018 clean shape: (30000, 86)  attack rate: 99.8%
Label distribution:
Label
DDoS attacks-LOIC-HTTP    29929
Benign                       71
Name: count, dtype: int64

  Cross-Year: All CICIDS2017 → CSE-CIC-IDS2018
  Train: 2,572,640   Test: 30,000
  Test attack rate: 99.8%
  Adding 51 missing columns as 0
  Dropping 56 extra columns
  Train features: 78   Test features after alignment: 78

  Logistic Regression Confusion Matrix:
 [[   27    44]
 [29929     0]]

  Random Forest Confusion Matrix:
 [[   71     0]
 [29929     0]]

Results:
              Model  Accuracy  Precision  Recall  F1  ROC_AUC
Logistic Regression    0.0009        0.0     0.0 0.0   0.0607
      Random Forest    0.0024        0.0  

## 7. Generalization Summary

| Test Scenario | Degree of Difficulty |
|---|---|
| Exp A: Web+DDoS → PortScan | Cross-attack-type, same year, same network |
| Exp B: All 2017 → Infiltration | Cross-attack-type, hardest (mimics benign) |
| Cross-year: 2017 → 2018 | Cross-year + cross-network (real-world deployment) |

Performance drop across these scenarios reveals how each factor hurts generalization.